# Session V: File I/O and Structured Data

Persist data using text files, JSON, and CSV, and learn how to design file-based mini databases.

## Exercise 1: Daily Journal App
**Objective:** Practice reading, writing, and appending to text files.

Build a simple daily journal application that stores entries in a plain text file (`journal.txt`). Each entry should include a timestamp and the user's text.

**Tasks:**
1. `add_entry(filename, text)` → Append a new journal entry with the current date/time to the file.
2. `read_journal(filename)` → Read and display all entries from the file. Handle the case where the file doesn't exist yet.
3. `search_journal(filename, keyword)` → Read the file and return only lines that contain the keyword (case-insensitive).
4. `count_entries(filename)` → Return the total number of entries in the journal.
5. `run_journal()` → A menu-driven interface: [A]dd entry, [R]ead all, [S]earch, [C]ount, [Q]uit.

In [2]:
from datetime import datetime

JOURNAL_FILE = "journal.txt"

def add_entry(filename, text):
    """Append a timestamped journal entry to the file."""
    # TODO: Open file in append mode
    # TODO: Write the current datetime + the entry text
    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M")
    with open(filename, "a") as file:
        file.write(f"[{timestamp}] {text}\n")
    print("Entry added.")

def read_journal(filename):
    """Read and print all entries. Handle FileNotFoundError."""
    # TODO: Try to open and read the file
    # TODO: Print each line (stripped)
    # TODO: Handle missing file gracefully
    try:
        with open(filename, "r") as file:
            for line in file:
                print(line.strip())
    except FileNotFoundError:
        print("No journal yet. Add your first entry.")

def search_journal(filename, keyword):
    """Return a list of lines containing the keyword (case-insensitive)."""
    # TODO: Read all lines
    # TODO: Filter lines that contain the keyword
    # TODO: Handle missing file
    keyword = keyword.lower()
    try:
        with open(filename, "r") as file:
            lines = file.readlines()
    except FileNotFoundError:
        print("No journal yet.")
        return []

    return [line.strip() for line in lines if keyword in line.lower()]

def count_entries(filename):
    """Count and return the number of journal entries."""
    # TODO: Read lines and count non-empty entries
    # TODO: Handle missing file (return 0)
    try:
        with open(filename, "r") as file:
            lines = file.readlines()
    except FileNotFoundError:
        return 0

    entries = [line for line in lines if line.strip() != ""]
    return len(entries)

def run_journal():
    """Menu-driven journal interface."""
    # TODO: while True loop with menu [A]dd [R]ead [S]earch [C]ount [Q]uit
    # TODO: Route to the correct function based on user input
    while True:
        print("[A]dd [R]ead [S]earch [C]ount [Q]uit")
        choice = input("Choose an option: ")
        choice = choice.lower().strip()

        if choice == "a":
            text = input("Write your entry: ")
            text = text.strip()
            if text == "":
                print("Entry cannot be empty.")
                continue
            add_entry(JOURNAL_FILE, text)
        elif choice == "r":
            read_journal(JOURNAL_FILE)
        elif choice == "s":
            keyword = input("Keyword to search: ")
            keyword = keyword.strip()
            results = search_journal(JOURNAL_FILE, keyword)
            if len(results) == 0:
                print("No entries match.")
            for line in results:
                print(line)
        elif choice == "c":
            print("Total entries:", count_entries(JOURNAL_FILE))
        elif choice == "q":
            print("Adios!")
            break
        else:
            print("Invalid option. Choose A, R, S, C or Q.")


# --- QUICK TEST (non-interactive) ---
# add_entry(JOURNAL_FILE, "Started learning about file I/O today!")
# add_entry(JOURNAL_FILE, "Python makes reading files so easy.")
# read_journal(JOURNAL_FILE)
# print("Search results:", search_journal(JOURNAL_FILE, "python"))
# print("Total entries:", count_entries(JOURNAL_FILE))

# run_journal()  # Uncomment to test the interactive menu

---
## Exercise 2: To-Do List Manager (JSON Persistence)
**Objective:** Build a CRUD application that stores data in a JSON file.

Create a to-do list manager where tasks are stored as a list of dictionaries in a `tasks.json` file. Each task has a `title`, a `priority` (high/medium/low), and a `done` status (True/False).

**Tasks:**
1. `load_tasks(filename)` → Load tasks from JSON. Return an empty list if the file is missing or corrupted.
2. `save_tasks(filename, tasks)` → Save the tasks list to JSON with pretty formatting.
3. `add_task(tasks)` → Prompt for title and priority, append a new task dict (done=False).
4. `list_tasks(tasks)` → Display all tasks in a formatted table with their index, status, title, and priority.
5. `mark_done(tasks)` → Ask for a task index and set its `done` field to True.
6. `delete_task(tasks)` → Ask for a task index and remove the task from the list.
7. `run_todo()` → Menu loop: [A]dd, [L]ist, [D]one, [R]emove, [Q]uit. Auto-save after every change.

**Data structure:**
```python
[
    {"title": "Buy groceries", "priority": "high", "done": False},
    {"title": "Read chapter 5", "priority": "medium", "done": True}
]
```

In [3]:
import json

TODO_FILE = "tasks.json"

def load_tasks(filename):
    """Load tasks from a JSON file. Return [] if missing or corrupted."""
    # TODO: try/except for FileNotFoundError and json.JSONDecodeError
    try:
        with open(filename, "r") as file:
            return json.load(file)
    except FileNotFoundError:
        print("No tasks file found. Starting fresh.")
        return []
    except json.JSONDecodeError:
        print("Tasks file is corrupted. Starting fresh.")
        return []

def save_tasks(filename, tasks):
    """Save tasks list to a JSON file with indent=2."""
    # TODO: Write tasks to file using json.dump
    with open(filename, "w") as file:
        json.dump(tasks, file, indent=2)

def add_task(tasks):
    """Prompt for title and priority, append new task."""
    # TODO: Get title from user (reject empty)
    # TODO: Get priority (validate: high/medium/low)
    # TODO: Append {"title": ..., "priority": ..., "done": False}
    while True:
        title = input("Task title: ")
        title = title.strip()
        if title == "":
            print("Title cannot be empty.")
            continue
        break

    while True:
        priority = input("Priority (high/medium/low): ")
        priority = priority.lower().strip()
        if priority not in ["high", "medium", "low"]:
            print("Priority must be high, medium or low.")
            continue
        break

    tasks.append({"title": title, "priority": priority, "done": False})
    print("Task added.")

def list_tasks(tasks):
    """Display all tasks in a formatted table."""
    # TODO: Handle empty list
    # TODO: Loop with enumerate, show index, done status, title, priority
    if len(tasks) == 0:
        print("No tasks yet.")
        return

    print(f"{'#':<4}{'Status':<8}{'Title':<30}{'Priority':<10}")
    for index, task in enumerate(tasks, start=1):
        status = "[x]" if task["done"] else "[ ]"
        print(f"{index:<4}{status:<8}{task['title']:<30}{task['priority']:<10}")

def get_task_index(tasks):
    """Ask for a task number and return its list index, or None if invalid."""
    try:
        number = int(input("Task number: "))
    except ValueError:
        print("Please type a number.")
        return None

    if number < 1 or number > len(tasks):
        print(f"Task number must be between 1 and {len(tasks)}.")
        return None

    return number - 1

def mark_done(tasks):
    """Ask for a task index and mark it as done."""
    # TODO: Validate the index (try/except + range check)
    # TODO: Set tasks[index]["done"] = True
    if len(tasks) == 0:
        print("No tasks yet.")
        return

    index = get_task_index(tasks)
    if index is None:
        return

    tasks[index]["done"] = True
    print(f"Task '{tasks[index]['title']}' marked as done.")

def delete_task(tasks):
    """Ask for a task index and remove the task."""
    # TODO: Validate the index
    # TODO: Remove with .pop()
    if len(tasks) == 0:
        print("No tasks yet.")
        return

    index = get_task_index(tasks)
    if index is None:
        return

    removed = tasks.pop(index)
    print(f"Task '{removed['title']}' removed.")

def run_todo():
    """Main menu loop with auto-save."""
    # TODO: Load tasks at start
    # TODO: while True menu: [A]dd [L]ist [D]one [R]emove [Q]uit
    # TODO: Save after every modification
    tasks = load_tasks(TODO_FILE)

    while True:
        print("[A]dd [L]ist [D]one [R]emove [Q]uit")
        choice = input("Choose an option: ")
        choice = choice.lower().strip()

        if choice == "a":
            add_task(tasks)
            save_tasks(TODO_FILE, tasks)
        elif choice == "l":
            list_tasks(tasks)
        elif choice == "d":
            mark_done(tasks)
            save_tasks(TODO_FILE, tasks)
        elif choice == "r":
            delete_task(tasks)
            save_tasks(TODO_FILE, tasks)
        elif choice == "q":
            print("Adios!")
            break
        else:
            print("Invalid option. Choose A, L, D, R or Q.")


# --- QUICK TEST (non-interactive) ---
# tasks = load_tasks(TODO_FILE)
# tasks.append({"title": "Test task", "priority": "high", "done": False})
# save_tasks(TODO_FILE, tasks)
# tasks = load_tasks(TODO_FILE)
# list_tasks(tasks)

run_todo()  # Uncomment to test the interactive menu

No tasks file found. Starting fresh.
[A]dd [L]ist [D]one [R]emove [Q]uit
Task added.
[A]dd [L]ist [D]one [R]emove [Q]uit
Task added.
[A]dd [L]ist [D]one [R]emove [Q]uit
#   Status  Title                         Priority  
1   [ ]     Buy food at mercadona         high      
2   [ ]     Go to the gym                 medium    
[A]dd [L]ist [D]one [R]emove [Q]uit
Task 'Buy food at mercadona' marked as done.
[A]dd [L]ist [D]one [R]emove [Q]uit
#   Status  Title                         Priority  
1   [x]     Buy food at mercadona         high      
2   [ ]     Go to the gym                 medium    
[A]dd [L]ist [D]one [R]emove [Q]uit
Task 'Go to the gym' removed.
[A]dd [L]ist [D]one [R]emove [Q]uit
#   Status  Title                         Priority  
1   [x]     Buy food at mercadona         high      
[A]dd [L]ist [D]one [R]emove [Q]uit
Adios!


---
## Exercise 3: CSV Expense Importer
**Objective:** Read, validate, and analyze data from a CSV file.

You receive a CSV file of monthly expenses. Some rows may have missing or invalid data. Write functions to import the data safely, clean it, and produce summary reports.

**First, run the cell below to generate the sample CSV file.** Then write the functions to process it.

**Tasks:**
1. `load_expenses(filename)` → Read the CSV using `csv.DictReader`. Validate each row: skip rows where `amount` is not a valid number or `category` is empty. Return a list of clean dicts (with `amount` converted to float).
2. `total_by_category(expenses)` → Return a dict mapping each category to its total spending.
3. `most_expensive(expenses)` → Return the single most expensive entry.
4. `print_report(expenses)` → Print a formatted summary: total spent, breakdown by category, and the most expensive item.

In [3]:
import csv

# --- RUN THIS CELL FIRST to generate the sample CSV ---
sample_data = """date,description,category,amount
2024-01-05,Grocery store,Food,52.30
2024-01-06,Bus pass,Transport,40.00
2024-01-07,Coffee shop,Food,4.50
2024-01-08,Netflix,Entertainment,15.99
2024-01-09,Gas station,Transport,bad_data
2024-01-10,Restaurant,Food,38.75
2024-01-11,Gym membership,,45.00
2024-01-12,Bookstore,Education,22.50
2024-01-13,Electricity bill,Utilities,89.00
2024-01-14,Movie tickets,Entertainment,24.00
2024-01-15,Taxi,Transport,18.50
2024-01-16,Online course,Education,49.99
2024-01-17,Supermarket,Food,67.20
2024-01-18,Water bill,Utilities,not_a_number
2024-01-19,Concert,Entertainment,75.00
2024-01-20,Pharmacy,Health,12.40"""

with open("expenses.csv", "w", newline="") as f:
    f.write(sample_data)

print("✅ expenses.csv created successfully.")

✅ expenses.csv created successfully.


In [4]:
import csv

def load_expenses(filename):
    """Load and validate expenses from CSV. Skip invalid rows."""
    # TODO: Open file with csv.DictReader
    # TODO: For each row, validate that 'amount' converts to float
    # TODO: Validate that 'category' is not empty
    # TODO: Return list of clean dicts with amount as float
    clean_expenses = []
    skipped = 0

    try:
        with open(filename, "r") as file:
            reader = csv.DictReader(file)
            for row in reader:
                try:
                    amount = float(row["amount"])
                except ValueError:
                    print(f"Skipped '{row['description']}': amount '{row['amount']}' is not a number.")
                    skipped = skipped + 1
                    continue

                category = row["category"].strip()
                if category == "":
                    print(f"Skipped '{row['description']}': category is empty.")
                    skipped = skipped + 1
                    continue

                clean_expenses.append({
                    "date": row["date"],
                    "description": row["description"],
                    "category": category,
                    "amount": amount
                })
    except FileNotFoundError:
        print(f"File '{filename}' not found.")
        return []

    print(f"Loaded {len(clean_expenses)} expenses. Skipped {skipped}.")
    return clean_expenses

def total_by_category(expenses):
    """Return {category: total_amount} dict."""
    # TODO: Accumulation pattern over categories
    totals = {}
    for expense in expenses:
        category = expense["category"]
        if category in totals:
            totals[category] = totals[category] + expense["amount"]
        else:
            totals[category] = expense["amount"]
    return totals

def most_expensive(expenses):
    """Return the expense dict with the highest amount."""
    # TODO: Use max() with a key, or loop
    if len(expenses) == 0:
        return None
    return max(expenses, key=lambda expense: expense["amount"])

def print_report(expenses):
    """Print a formatted expense summary report."""
    # TODO: Total spent
    # TODO: Breakdown by category
    # TODO: Most expensive item
    if len(expenses) == 0:
        print("No expenses to report.")
        return

    total = sum([expense["amount"] for expense in expenses])
    totals = total_by_category(expenses)
    top = most_expensive(expenses)

    print("\n===== EXPENSE REPORT =====")
    print(f"Total spent: {total:.2f}€")

    print("\nBy category:")
    for category in sorted(totals, key=lambda category: totals[category], reverse=True):
        print(f"  {category:<15}{totals[category]:>8.2f}€")

    print(f"\nMost expensive: {top['description']} ({top['category']}) - {top['amount']:.2f}€ on {top['date']}")


# --- TEST ---
expenses = load_expenses("expenses.csv")
print_report(expenses)

Skipped 'Gas station': amount 'bad_data' is not a number.
Skipped 'Gym membership': category is empty.
Skipped 'Water bill': amount 'not_a_number' is not a number.
Loaded 13 expenses. Skipped 3.

===== EXPENSE REPORT =====
Total spent: 510.13€

By category:
  Food             162.75€
  Entertainment    114.99€
  Utilities         89.00€
  Education         72.49€
  Transport         58.50€
  Health            12.40€

Most expensive: Electricity bill (Utilities) - 89.00€ on 2024-01-13


---
## Exercise 4: File Content Validator
**Objective:** Write robust validation functions for both JSON and CSV files.

You will build a validation toolkit that checks whether files are well-formed and contain the expected structure *before* your application tries to use the data.

**First, run the cell below to generate sample test files** (some valid, some intentionally broken). Then write the validators.

**Tasks:**
1. `validate_json(filename, required_keys)` → Check that the file (a) exists, (b) contains valid JSON, (c) is a list of dicts, and (d) each dict contains all `required_keys`. Return a report dict: `{"valid": int, "invalid": int, "errors": [str]}`.
2. `validate_csv(filename, expected_columns)` → Check that the file (a) exists, (b) has the correct number of columns in the header, and (c) each data row has the correct number of fields. Return a similar report dict.
3. `print_validation_report(report, filename)` → Pretty-print the validation results.

In [5]:
import json

# --- RUN THIS CELL FIRST to generate test files ---

# Good JSON file
good_json = [
    {"name": "Alice", "age": 22, "email": "alice@test.com"},
    {"name": "Bob", "age": 25, "email": "bob@test.com"},
    {"name": "Charlie", "age": 30, "email": "charlie@test.com"},
]
with open("good_data.json", "w") as f:
    json.dump(good_json, f, indent=2)

# Bad JSON file (missing keys, wrong types)
bad_json = [
    {"name": "Alice", "age": 22, "email": "alice@test.com"},
    {"name": "Bob", "age": 25},
    "this is not a dict",
    {"name": "Diana", "age": "not_a_number", "email": "diana@test.com"},
]
with open("bad_data.json", "w") as f:
    json.dump(bad_json, f, indent=2)

# Corrupted JSON file
with open("corrupted.json", "w") as f:
    f.write('{"name": "broken", missing_quotes: true}')

# Good CSV file
with open("good_data.csv", "w", newline="") as f:
    f.write("name,age,email\n")
    f.write("Alice,22,alice@test.com\n")
    f.write("Bob,25,bob@test.com\n")

# Bad CSV file (inconsistent columns)
with open("bad_data.csv", "w", newline="") as f:
    f.write("name,age,email\n")
    f.write("Alice,22,alice@test.com\n")
    f.write("Bob,25\n")
    f.write("Charlie,30,charlie@test.com,extra_field\n")
    f.write("Diana,28,diana@test.com\n")

print("✅ All test files created successfully.")

✅ All test files created successfully.


In [6]:
import json
import csv

def validate_json(filename, required_keys):
    """Validate a JSON file's structure and required keys.
    Return {"valid": int, "invalid": int, "errors": [str]}."""
    # TODO: Try to open and parse the file
    # TODO: Check that root is a list
    # TODO: Check each item is a dict with all required_keys
    # TODO: Collect errors and counts
    report = {"valid": 0, "invalid": 0, "errors": []}

    try:
        with open(filename, "r") as file:
            data = json.load(file)
    except FileNotFoundError:
        report["errors"].append("File not found.")
        return report
    except json.JSONDecodeError:
        report["errors"].append("File contains invalid JSON.")
        return report

    if not isinstance(data, list):
        report["errors"].append("Root element is not a list.")
        return report

    for index, item in enumerate(data):
        if not isinstance(item, dict):
            report["invalid"] = report["invalid"] + 1
            report["errors"].append(f"Item {index}: not a dictionary.")
            continue

        missing = [key for key in required_keys if key not in item]
        if len(missing) > 0:
            report["invalid"] = report["invalid"] + 1
            report["errors"].append(f"Item {index}: missing keys {missing}.")
            continue

        report["valid"] = report["valid"] + 1

    return report

def validate_csv(filename, expected_columns):
    """Validate a CSV file's structure and column count.
    Return {"valid": int, "invalid": int, "errors": [str]}."""
    # TODO: Try to open the file
    # TODO: Check header column count
    # TODO: Check each row's column count
    # TODO: Collect errors and counts
    report = {"valid": 0, "invalid": 0, "errors": []}

    try:
        with open(filename, "r") as file:
            reader = csv.reader(file)
            header = next(reader)

            if len(header) != expected_columns:
                report["errors"].append(f"Header has {len(header)} columns, expected {expected_columns}.")
                return report

            for line_number, row in enumerate(reader, start=2):
                if len(row) != expected_columns:
                    report["invalid"] = report["invalid"] + 1
                    report["errors"].append(f"Line {line_number}: has {len(row)} fields, expected {expected_columns}.")
                    continue
                report["valid"] = report["valid"] + 1
    except FileNotFoundError:
        report["errors"].append("File not found.")
        return report

    return report

def print_validation_report(report, filename):
    """Pretty-print the validation results."""
    # TODO: Print filename, valid/invalid counts, and error details
    print(f"Validation report for '{filename}'")
    print(f"  Valid rows:   {report['valid']}")
    print(f"  Invalid rows: {report['invalid']}")

    if len(report["errors"]) == 0:
        print("  No errors found.")
    else:
        print("  Errors:")
        for error in report["errors"]:
            print(f"    - {error}")


# --- TEST ---
print("=" * 50)
report = validate_json("good_data.json", ["name", "age", "email"])
print_validation_report(report, "good_data.json")

print("=" * 50)
report = validate_json("bad_data.json", ["name", "age", "email"])
print_validation_report(report, "bad_data.json")

print("=" * 50)
report = validate_json("corrupted.json", ["name", "age", "email"])
print_validation_report(report, "corrupted.json")

print("=" * 50)
report = validate_csv("good_data.csv", 3)
print_validation_report(report, "good_data.csv")

print("=" * 50)
report = validate_csv("bad_data.csv", 3)
print_validation_report(report, "bad_data.csv")

Validation report for 'good_data.json'
  Valid rows:   3
  Invalid rows: 0
  No errors found.
Validation report for 'bad_data.json'
  Valid rows:   2
  Invalid rows: 2
  Errors:
    - Item 1: missing keys ['email'].
    - Item 2: not a dictionary.
Validation report for 'corrupted.json'
  Valid rows:   0
  Invalid rows: 0
  Errors:
    - File contains invalid JSON.
Validation report for 'good_data.csv'
  Valid rows:   2
  Invalid rows: 0
  No errors found.
Validation report for 'bad_data.csv'
  Valid rows:   2
  Invalid rows: 2
  Errors:
    - Line 3: has 2 fields, expected 3.
    - Line 4: has 4 fields, expected 3.


---
## Exercise 5: Format Converter & Exporter
**Objective:** Convert data between JSON and CSV formats in both directions.

Build a bidirectional converter that can transform data files between JSON and CSV. This is an extremely common real-world task — APIs return JSON, but spreadsheets need CSV, and vice versa.

**Tasks:**
1. `json_to_csv(json_filename, csv_filename)` → Read a JSON file (list of dicts) and write it as a CSV file. Use the keys of the first dict as column headers.
2. `csv_to_json(csv_filename, json_filename)` → Read a CSV file and write it as a JSON file (list of dicts).
3. `export_filtered_csv(json_filename, csv_filename, filter_key, filter_value)` → Read a JSON file, keep only records where `record[filter_key] == filter_value`, and export the filtered data to CSV.
4. Handle edge cases: empty files, missing files, and mismatched structures.

In [7]:
import json
import csv

# --- Setup: create a test JSON file ---
students = [
    {"name": "Alice", "age": 22, "city": "Madrid", "grade": "A"},
    {"name": "Bob", "age": 25, "city": "London", "grade": "B"},
    {"name": "Charlie", "age": 30, "city": "Paris", "grade": "A"},
    {"name": "Diana", "age": 28, "city": "Madrid", "grade": "C"},
    {"name": "Eve", "age": 23, "city": "London", "grade": "A"},
]

with open("students.json", "w") as f:
    json.dump(students, f, indent=2)

print("✅ students.json created.")

✅ students.json created.


In [8]:
import json
import csv

def json_to_csv(json_filename, csv_filename):
    """Convert a JSON file (list of dicts) to a CSV file."""
    # TODO: Load JSON data
    # TODO: Extract fieldnames from the first dict's keys
    # TODO: Write CSV with DictWriter
    # TODO: Handle errors (FileNotFoundError, empty data)
    try:
        with open(json_filename, "r") as file:
            data = json.load(file)
    except FileNotFoundError:
        print(f"File '{json_filename}' not found.")
        return
    except json.JSONDecodeError:
        print(f"File '{json_filename}' contains invalid JSON.")
        return

    if not isinstance(data, list) or len(data) == 0:
        print(f"File '{json_filename}' has no records to convert.")
        return

    fieldnames = list(data[0].keys())

    with open(csv_filename, "w", newline="") as file:
        writer = csv.DictWriter(file, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(data)

    print(f"Converted {len(data)} records: {json_filename} -> {csv_filename}")

def csv_to_json(csv_filename, json_filename):
    """Convert a CSV file to a JSON file (list of dicts)."""
    # TODO: Read CSV with DictReader
    # TODO: Convert to list of dicts
    # TODO: Write JSON
    # TODO: Handle errors
    try:
        with open(csv_filename, "r") as file:
            reader = csv.DictReader(file)
            data = list(reader)
    except FileNotFoundError:
        print(f"File '{csv_filename}' not found.")
        return

    if len(data) == 0:
        print(f"File '{csv_filename}' has no records to convert.")
        return

    with open(json_filename, "w") as file:
        json.dump(data, file, indent=2)

    print(f"Converted {len(data)} records: {csv_filename} -> {json_filename}")

def export_filtered_csv(json_filename, csv_filename, filter_key, filter_value):
    """Export only records where record[filter_key] == filter_value to CSV."""
    # TODO: Load JSON
    # TODO: Filter records
    # TODO: Write filtered data to CSV
    # TODO: Handle empty results and errors
    try:
        with open(json_filename, "r") as file:
            data = json.load(file)
    except FileNotFoundError:
        print(f"File '{json_filename}' not found.")
        return
    except json.JSONDecodeError:
        print(f"File '{json_filename}' contains invalid JSON.")
        return

    if not isinstance(data, list) or len(data) == 0:
        print(f"File '{json_filename}' has no records.")
        return

    if filter_key not in data[0]:
        print(f"Key '{filter_key}' does not exist in the records.")
        return

    filtered = [record for record in data if record[filter_key] == filter_value]

    if len(filtered) == 0:
        print(f"No records where {filter_key} = {filter_value}. Nothing exported.")
        return

    fieldnames = list(filtered[0].keys())
    with open(csv_filename, "w", newline="") as file:
        writer = csv.DictWriter(file, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(filtered)

    print(f"Exported {len(filtered)} records where {filter_key} = {filter_value} -> {csv_filename}")


# --- TEST ---
# JSON → CSV
json_to_csv("students.json", "students_export.csv")

# CSV → JSON (round-trip test)
csv_to_json("students_export.csv", "students_roundtrip.json")

# Filtered export: only students from Madrid
export_filtered_csv("students.json", "madrid_students.csv", "city", "Madrid")

# Verify round-trip
with open("students_roundtrip.json", "r") as f:
    roundtrip = json.load(f)
print(f"\nRound-trip data ({len(roundtrip)} records):")
for r in roundtrip:
    print(f"  {r}")

Converted 5 records: students.json -> students_export.csv
Converted 5 records: students_export.csv -> students_roundtrip.json
Exported 2 records where city = Madrid -> madrid_students.csv

Round-trip data (5 records):
  {'name': 'Alice', 'age': '22', 'city': 'Madrid', 'grade': 'A'}
  {'name': 'Bob', 'age': '25', 'city': 'London', 'grade': 'B'}
  {'name': 'Charlie', 'age': '30', 'city': 'Paris', 'grade': 'A'}
  {'name': 'Diana', 'age': '28', 'city': 'Madrid', 'grade': 'C'}
  {'name': 'Eve', 'age': '23', 'city': 'London', 'grade': 'A'}


---
## Exercise 6: Student Records Database (Capstone)
**Objective:** Build a complete file-based CRUD application combining JSON persistence, CSV import/export, and data validation.

Create a student records management system that stores data in a JSON file and supports importing/exporting CSV. This exercise combines **every concept** from this session.

**Required features:**
1. **[A]dd Student** — Prompt for name, age, email, and grade. Validate all inputs (age must be a positive integer, email must contain `@`, grade must be A/B/C/D/F). Store in JSON.
2. **[L]ist Students** — Display all students in a formatted table.
3. **[S]earch** — Find a student by name (case-insensitive partial match).
4. **[U]pdate** — Modify a student's grade or email by name.
5. **[D]elete** — Remove a student by name.
6. **[I]mport CSV** — Load students from a CSV file, validating each row before adding. Skip invalid rows and report what was skipped.
7. **[E]xport CSV** — Export all current students to a CSV file.
8. **[Q]uit** — Exit the program.

**Data structure (JSON):**
```python
[
    {"name": "Alice", "age": 22, "email": "alice@uni.edu", "grade": "A"},
    {"name": "Bob", "age": 25, "email": "bob@uni.edu", "grade": "B"}
]
```

In [ ]:
import json
import csv

DB_FILE = "students_db.json"

# --- PERSISTENCE ---

def load_db(filename):
    """Load student records from JSON. Return [] if missing/corrupted."""
    # TODO: try/except for FileNotFoundError and json.JSONDecodeError
    try:
        with open(filename, "r") as file:
            return json.load(file)
    except FileNotFoundError:
        print("No database found. Starting fresh.")
        return []
    except json.JSONDecodeError:
        print("Database file is corrupted. Starting fresh.")
        return []

def save_db(filename, students):
    """Save student records to JSON."""
    # TODO: json.dump with indent=2
    with open(filename, "w") as file:
        json.dump(students, file, indent=2)

# --- VALIDATION ---

def validate_student(name, age, email, grade):
    """Validate student fields. Return (is_valid, error_message)."""
    # TODO: name must be non-empty
    # TODO: age must be a positive integer
    # TODO: email must contain '@'
    # TODO: grade must be one of A, B, C, D, F
    if name.strip() == "":
        return (False, "Name cannot be empty.")

    try:
        age = int(age)
    except ValueError:
        return (False, f"Age '{age}' is not a number.")

    if age <= 0:
        return (False, "Age must be a positive number.")

    if "@" not in email:
        return (False, f"Email '{email}' must contain @.")

    if grade.upper() not in ["A", "B", "C", "D", "F"]:
        return (False, f"Grade '{grade}' must be A, B, C, D or F.")

    return (True, "")

def find_student(students, name):
    """Return the student dict with this exact name (case-insensitive), or None."""
    for student in students:
        if student["name"].lower() == name.lower():
            return student
    return None

# --- CRUD OPERATIONS ---

def add_student(students):
    """Prompt for student info, validate, and add to list."""
    # TODO: Get input, call validate_student, append if valid
    name = input("Name: ")
    name = name.strip()
    age = input("Age: ")
    age = age.strip()
    email = input("Email: ")
    email = email.strip()
    grade = input("Grade (A/B/C/D/F): ")
    grade = grade.strip().upper()

    is_valid, error = validate_student(name, age, email, grade)
    if not is_valid:
        print("Not added:", error)
        return

    if find_student(students, name) is not None:
        print(f"Not added: '{name}' already exists.")
        return

    students.append({"name": name, "age": int(age), "email": email, "grade": grade})
    print(f"Student '{name}' added.")

def list_students(students):
    """Display all students in a formatted table."""
    # TODO: Handle empty list
    # TODO: Print formatted table with headers
    if len(students) == 0:
        print("No students in the database.")
        return

    print(f"{'Name':<15}{'Age':<6}{'Email':<25}{'Grade':<6}")
    print("-" * 52)
    for student in students:
        print(f"{student['name']:<15}{student['age']:<6}{student['email']:<25}{student['grade']:<6}")

def search_student(students):
    """Search by name (case-insensitive partial match)."""
    # TODO: Get search term
    # TODO: Filter students whose name contains the term
    term = input("Search name: ")
    term = term.lower().strip()

    results = [student for student in students if term in student["name"].lower()]

    if len(results) == 0:
        print("No students match.")
        return

    list_students(results)

def update_student(students):
    """Update a student's grade or email by name."""
    # TODO: Find student by name
    # TODO: Ask which field to update
    # TODO: Validate new value before applying
    name = input("Student name to update: ")
    name = name.strip()

    student = find_student(students, name)
    if student is None:
        print(f"Student '{name}' not found.")
        return

    field = input("Update [G]rade or [E]mail? ")
    field = field.lower().strip()

    if field == "g":
        new_grade = input("New grade (A/B/C/D/F): ")
        new_grade = new_grade.strip().upper()
        is_valid, error = validate_student(student["name"], student["age"], student["email"], new_grade)
        if not is_valid:
            print("Not updated:", error)
            return
        student["grade"] = new_grade
        print(f"Grade updated to {new_grade}.")
    elif field == "e":
        new_email = input("New email: ")
        new_email = new_email.strip()
        is_valid, error = validate_student(student["name"], student["age"], new_email, student["grade"])
        if not is_valid:
            print("Not updated:", error)
            return
        student["email"] = new_email
        print(f"Email updated to {new_email}.")
    else:
        print("Invalid option. Choose G or E.")

def delete_student(students):
    """Remove a student by name."""
    # TODO: Find and remove student, confirm before deleting
    name = input("Student name to delete: ")
    name = name.strip()

    student = find_student(students, name)
    if student is None:
        print(f"Student '{name}' not found.")
        return

    confirm = input(f"Delete '{student['name']}'? (yes/no): ")
    confirm = confirm.lower().strip()
    if confirm != "yes":
        print("Delete cancelled.")
        return

    students.remove(student)
    print(f"Student '{student['name']}' deleted.")

# --- IMPORT / EXPORT ---

def import_csv(students, csv_filename):
    """Import students from CSV, validating each row."""
    # TODO: Read CSV with DictReader
    # TODO: Validate each row with validate_student
    # TODO: Skip invalid rows, print what was skipped
    # TODO: Append valid students to the list
    try:
        with open(csv_filename, "r") as file:
            reader = csv.DictReader(file)
            rows = list(reader)
    except FileNotFoundError:
        print(f"File '{csv_filename}' not found.")
        return

    added = 0
    skipped = 0
    for line_number, row in enumerate(rows, start=2):
        name = row["name"].strip()
        age = row["age"].strip()
        email = row["email"].strip()
        grade = row["grade"].strip().upper()

        is_valid, error = validate_student(name, age, email, grade)
        if not is_valid:
            print(f"  Line {line_number} skipped: {error}")
            skipped = skipped + 1
            continue

        if find_student(students, name) is not None:
            print(f"  Line {line_number} skipped: '{name}' already exists.")
            skipped = skipped + 1
            continue

        students.append({"name": name, "age": int(age), "email": email, "grade": grade})
        added = added + 1

    print(f"Imported {added} student(s). Skipped {skipped}.")

def export_csv(students, csv_filename):
    """Export all students to a CSV file."""
    # TODO: Write CSV with DictWriter
    # TODO: Handle empty data
    if len(students) == 0:
        print("No students to export.")
        return

    with open(csv_filename, "w", newline="") as file:
        writer = csv.DictWriter(file, fieldnames=["name", "age", "email", "grade"])
        writer.writeheader()
        writer.writerows(students)

    print(f"Exported {len(students)} student(s) to {csv_filename}.")

# --- MAIN MENU ---

def run_student_db():
    """Main menu loop."""
    students = load_db(DB_FILE)
    print(f"Loaded {len(students)} student(s) from database.")

    while True:
        print("\n===== STUDENT RECORDS DATABASE =====")
        print("[A]dd  [L]ist  [S]earch  [U]pdate  [D]elete")
        print("[I]mport CSV  [E]xport CSV  [Q]uit")
        choice = input("> ").strip().lower()

        if choice == "a":
            add_student(students)
            save_db(DB_FILE, students)
        elif choice == "l":
            list_students(students)
        elif choice == "s":
            search_student(students)
        elif choice == "u":
            update_student(students)
            save_db(DB_FILE, students)
        elif choice == "d":
            delete_student(students)
            save_db(DB_FILE, students)
        elif choice == "i":
            csv_file = input("CSV filename to import: ").strip()
            import_csv(students, csv_file)
            save_db(DB_FILE, students)
        elif choice == "e":
            csv_file = input("CSV filename to export to: ").strip()
            export_csv(students, csv_file)
        elif choice == "q":
            save_db(DB_FILE, students)
            print("Database saved. Goodbye!")
            break
        else:
            print("Invalid option. Please try again.")

run_student_db()  # Uncomment to test

No database found. Starting fresh.
Loaded 0 student(s) from database.

===== STUDENT RECORDS DATABASE =====
[A]dd  [L]ist  [S]earch  [U]pdate  [D]elete
[I]mport CSV  [E]xport CSV  [Q]uit
Invalid option. Please try again.

===== STUDENT RECORDS DATABASE =====
[A]dd  [L]ist  [S]earch  [U]pdate  [D]elete
[I]mport CSV  [E]xport CSV  [Q]uit
Not added: Name cannot be empty.

===== STUDENT RECORDS DATABASE =====
[A]dd  [L]ist  [S]earch  [U]pdate  [D]elete
[I]mport CSV  [E]xport CSV  [Q]uit


In [10]:
# --- Generate a sample CSV for testing the Import feature ---

sample_import = """name,age,email,grade
Frank,21,frank@uni.edu,B
Grace,23,grace@uni.edu,A
Hank,not_a_number,hank@uni.edu,C
Ivy,20,ivy_no_at_sign,A
Jack,22,jack@uni.edu,Z
Karen,24,karen@uni.edu,B"""

with open("import_students.csv", "w", newline="") as f:
    f.write(sample_import)

print("import_students.csv created (3 valid, 3 invalid rows).")

import_students.csv created (3 valid, 3 invalid rows).
